# KisanAI - Agricultural RAG

KisanAI / Sampoorn Kisan AI Sahayak

## Goal
Build a small agricultural RAG prototype.
RAG is not trained like Random Forest. It indexes documents, retrieves relevant passages, and uses an LLM to formulate an answer.

In [ ]:
!pip -q install sentence-transformers faiss-cpu pypdf

In [ ]:
from google.colab import files
uploaded = files.upload()

print("Upload TXT/PDF agricultural documents, government scheme documents, crop guides, etc.")

In [ ]:
import os, glob

uploaded_files = list(uploaded.keys())
print("Uploaded:", uploaded_files)

In [ ]:
# Extract text from PDFs and TXT files
from pypdf import PdfReader

documents = []

for path in uploaded_files:
    if path.lower().endswith(".pdf"):
        reader = PdfReader(path)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        documents.append({"source": path, "text": text})
    elif path.lower().endswith(".txt"):
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            documents.append({"source": path, "text": f.read()})

print("Documents:", len(documents))
for d in documents:
    print(d["source"], len(d["text"]))

In [ ]:
# Simple chunking
chunks = []
chunk_size = 800
overlap = 100

for doc in documents:
    text = doc["text"]
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append({
            "source": doc["source"],
            "text": text[start:end]
        })
        start += chunk_size - overlap

print("Chunks:", len(chunks))
print(chunks[0]["text"][:500] if chunks else "No chunks")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in chunks]
if texts:
    embeddings = embedder.encode(texts, normalize_embeddings=True)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(np.asarray(embeddings, dtype="float32"))
    print("Vector index ready:", index.ntotal)
else:
    print("Upload at least one TXT/PDF document.")

In [ ]:
def retrieve(question, k=3):
    q = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, min(k, len(chunks)))
    return [
        {"score": float(scores[0][i]), **chunks[int(idx)]}
        for i, idx in enumerate(ids[0]) if idx >= 0
    ]

question = "What support or farming advice is available for a farmer?"
results = retrieve(question)

for r in results:
    print("\nSOURCE:", r["source"], "SCORE:", round(r["score"], 3))
    print(r["text"][:700])

## LLM generation
The retrieval layer above is the core RAG component. For the final answer, connect the retrieved context to your chosen LLM provider from your application backend. Keep API keys out of notebooks and source control.